# Disney AI: Dynamic Collaboration — Guided Demo

**Goal**: Run an end-to-end pass — agent negotiation → orchestration plan → visuals.

This notebook is **self-contained**. It writes outputs into an `outputs/` folder so your JS conductor (Chart.js) can read `orchestration_plan.csv`.

## 0. Setup

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

OUTPUT = "outputs"
os.makedirs(OUTPUT, exist_ok=True)
np.random.seed(42)
print("Outputs directory:", os.path.abspath(OUTPUT))

## 1. Define Agents & Utility Functions

In [ ]:
agents = ["Music","Animation","Dialogue","Story"]
weights = {
    "Music":     dict(spot=0.45, time=0.35, emot=0.20),
    "Animation": dict(spot=0.30, time=0.50, emot=0.20),
    "Dialogue":  dict(spot=0.25, time=0.35, emot=0.40),
    "Story":     dict(spot=0.40, time=0.30, emot=0.30),
}

def utility(agent, spotlight, timing, emotion):
    w = weights[agent]
    return w["spot"]*spotlight + w["time"]*timing + w["emot"]*emotion

print("Agents:", agents)

## 2. Game-Theory Negotiation (placeholder best-response loop)

In [ ]:
T = 60  # time steps
trace = []
spot, tim, emo = 0.5, 0.5, 0.5

for t in range(T):
    for a in agents:
        ds, dt, de = np.random.normal(0, 0.05, 3)
        spot = np.clip(spot + ds, 0, 1)
        tim  = np.clip(tim  + dt, 0, 1)
        emo  = np.clip(emo  + de, 0, 1)
        u = utility(a, spot, tim, emo)
        trace.append(dict(t=t, agent=a, spot=spot, time=tim, emot=emo, util=u))

df = pd.DataFrame(trace)
df.to_csv(f"{OUTPUT}/negotiation_trace.csv", index=False)
df.head()

## 3. Derive Orchestration Plan (tempo curve + lead agent per step)

In [ ]:
dominant = df.groupby("t")["util"].idxmax()
winners = df.loc[dominant, "agent"].to_numpy()
tempo = 80 + 40*df.groupby("t")["time"].mean().to_numpy()

plan = pd.DataFrame({"t": range(len(winners)), "tempo_bpm": tempo, "lead": winners})
plan.to_csv(f"{OUTPUT}/orchestration_plan.csv", index=False)
plan.head()

## 4. Visuals (one chart per figure)

In [ ]:
plt.figure()
plt.plot(plan["t"], plan["tempo_bpm"])
plt.title("Tempo Curve (Negotiated)")
plt.xlabel("t"); plt.ylabel("BPM")
plt.savefig(f"{OUTPUT}/tempo_curve.png", dpi=160)
plt.show()

## 5. Next Steps
- Replace this Python placeholder negotiation with your Java/Kotlin engine.
- Point the JS Chart.js conductor at `outputs/orchestration_plan.csv`.
- Integrate real character/scene datasets when available.